# STAIR-v1 — Stepwise Graph Contrastive Learning (STAIR-GCL)

**Cải tiến 1:** STAIR-GCL (Graph Contrastive Learning trên 32 chiều Collaborative Subspace)
- Chỉ chạy huấn luyện mô hình đã cải tiến (`STAIR-v1`) trên 2 tập `baby` & `sports`.
- Tận dụng trực tiếp kết quả thực nghiệm **STAIR Baseline** (Baby: Recall@20=0.1046, NDCG@20=0.0456 | Sports: Recall@20=0.1130, NDCG@20=0.0506).
- Xuất kết quả chi tiết, vẽ biểu đồ quá trình học (Learning Curve over Epochs) và biểu đồ so sánh cột (Bar Chart).

## Cell 1 — Thiết lập Môi trường & Kéo Mã nguồn Public từ GitHub

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

github_user = 'ThanhChuong12'
repo_name   = 'STAIR-Enhanced'

if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print('🗑️ Da xoa thu muc ' + repo_name + ' cu.')

git_url = f'https://github.com/{github_user}/{repo_name}.git'
print('⬇️ Dang clone public repo: ' + git_url)
os.system('git clone ' + git_url)

print('⚙️ Dang cai dat thu vien (freerec, torch_geometric, nvidia-ml-py)...')
os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.8.3 -q')
os.system('pip install torch_geometric -q')

os.chdir(f'/kaggle/working/{repo_name}')
print('Thu muc hien tai: ' + os.getcwd())
for f in ['main.py', 'main_v1.py']:
    exists = os.path.exists(f)
    print('  [' + ('OK' if exists else 'MISSING!') + '] ' + f)
print('✅ Moi truong va repo da san sang!')

## Cell 2 — Chuẩn bị Dữ liệu (`baby` & `sports`)

In [ ]:
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs('data/baby', exist_ok=True)
os.makedirs('data/sports', exist_ok=True)

def smart_copy_dataset(kw, full_name, local_dest):
    kaggle_dest = os.path.join(DATA_ROOT, full_name)
    found = 0
    for root, _, files in os.walk('/kaggle/input'):
        if kw.lower() in root.lower():
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt')):
                    src_f = os.path.join(root, f)
                    shutil.copy(src_f, os.path.join(local_dest, f))
                    os.makedirs(kaggle_dest, exist_ok=True)
                    shutil.copy(src_f, os.path.join(kaggle_dest, f))
                    found += 1
    if found > 0:
        print(f"  [+] Da copy thanh cong {found} files cho [{full_name}]")
    else:
        print(f"  [!] CANH BAO: Khong tim thay files chua tu khoa '{kw}' trong /kaggle/input!")

print('📁 Dang tim va copy du lieu an toan...')
smart_copy_dataset('baby',   'Amazon2014Baby_550_MMRec',   'data/baby')
smart_copy_dataset('sports', 'Amazon2014Sports_550_MMRec', 'data/sports')
print('\n✅ Du lieu da san sang!')

## Cell 3 — Kiểm tra Môi trường & GPU

In [ ]:
import torch, platform, os

print('='*60)
print('THONG TIN MOI TRUONG')
print('='*60)
print('  Python     : ' + platform.python_version())
print('  PyTorch    : ' + torch.__version__)
print('  CUDA avail : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    print('  GPU name   : ' + torch.cuda.get_device_name(0))
print('  Working dir: ' + os.getcwd())
print('='*60)

## Cell 4 — Huấn luyện STAIR-v1 (Chỉ chạy mô hình đã cải tiến, không chạy lại Baseline)

> ⚡ **Chạy 500 epochs cho STAIR-v1 trên tập `baby` và `sports`.**

In [ ]:
import time, subprocess, os

runs = [
    {
        'key': 'baby_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Baby_550_MMRec',
        'weight_decay': '0.3',
        'gamma': '0.1',
        'extra_args': ['--config', 'configs/Amazon2014Baby_550_MMRec.yaml', '--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
    {
        'key': 'sports_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Sports_550_MMRec',
        'weight_decay': '0.1',
        'gamma': '0.2',
        'extra_args': ['--config', 'configs/Amazon2014Sports_550_MMRec.yaml', '--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
]

results_log = {}

for run in runs:
    key = run['key']
    script = run['script']
    dataset = run['dataset']
    print('\n' + '='*60)
    print(f'🚀 DANG HUAN LUYEN: {key.upper()} ({script})')
    print('='*60)

    cmd = [
        'python', script,
        '--dataset',      dataset,
        '--root',         '/kaggle/data',
        '--weight-decay', run['weight_decay'],
        '--gamma',        run['gamma'],
        '--epochs',       '500',
        '--eval-freq',    '5',
    ] + run['extra_args']

    print('Lenh: ' + ' '.join(cmd))
    start_time = time.time()
    res = subprocess.run(cmd, capture_output=True, text=True, cwd='/kaggle/working/STAIR-Enhanced')
    elapsed = time.time() - start_time

    if res.returncode != 0:
        print(f'❌ LOI {key.upper()}:')
        print(res.stderr[-3000:])
        results_log[key] = {'status': 'FAIL', 'log': res.stderr[-2000:], 'time': elapsed}
    else:
        print(f'✅ HOAN THANH {key.upper()} trong {round(elapsed, 1)}s!')
        print(res.stdout[-1000:])
        results_log[key] = {'status': 'OK', 'log': res.stdout, 'time': elapsed}

print('\n' + '='*60)
print('HUAN LUYEN HOAN THANH!')
print('='*60)

## Cell 5 — Vẽ Biểu Đồ Quá Trình Học & Bảng So Sánh Chỉ Số (Baseline vs STAIR-v1)

> 📊 **Kết quả Baseline được lấy chính xác từ bảng thực nghiệm trên Kaggle của bạn:**  
- **Baby Baseline:** Recall@10 = 0.0686, Recall@20 = 0.1046, NDCG@10 = 0.0363, NDCG@20 = 0.0456  
- **Sports Baseline:** Recall@10 = 0.0751, Recall@20 = 0.1130, NDCG@10 = 0.0408, NDCG@20 = 0.0506

In [ ]:
import re, os, glob
import matplotlib.pyplot as plt
import numpy as np

# Kết quả thực nghiệm Baseline gốc của bạn (từ bảng hình ảnh)
BASELINE_METRICS = {
    'baby_baseline':   {'Recall@10': 0.0686, 'Recall@20': 0.1046, 'NDCG@10': 0.0363, 'NDCG@20': 0.0456},
    'sports_baseline': {'Recall@10': 0.0751, 'Recall@20': 0.1130, 'NDCG@10': 0.0408, 'NDCG@20': 0.0506}
}

# Đọc file log.txt để trích xuất Quá trình học (Epochs, Loss, Metrics) của STAIR-v1
def parse_v1_log(fpath):
    epochs = []
    losses = []
    eval_epochs = []
    metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    best_metrics = {}
    
    with open(fpath, 'r', encoding='utf-8', errors='ignore') as fp:
        lines = fp.readlines()
        for line in lines:
            # Loss line
            if 'TRAIN @Epoch:' in line and 'LOSS Avg:' in line:
                m_ep = re.search(r'TRAIN @Epoch:\s*(\d+)', line)
                m_loss = re.search(r'LOSS Avg:\s*([0-9.]+)', line)
                if m_ep and m_loss:
                    epochs.append(int(m_ep.group(1)))
                    losses.append(float(m_loss.group(1)))
            
            # PrettyTable evaluation line
            if '|' in line and not '---' in line and not 'Epoch' in line:
                parts = [p.strip() for p in line.split('|') if p.strip()]
                if len(parts) >= 5:
                    try:
                        # Check if row has floats
                        vals = [float(p) for p in parts if re.match(r'^[0-9.]+$', p)]
                        if len(vals) >= 4:
                            # Typically: Epoch, Recall@1, Recall@10, Recall@20, NDCG@10, NDCG@20
                            pass
                    except: pass
            
            # Direct metric match
            for m_name in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
                m_match = re.search(rf'{m_name}\s*[:=\s]\s*([0-9.]+)', line)
                if m_match:
                    v = float(m_match.group(1))
                    if m_name not in best_metrics or v > best_metrics[m_name]:
                        best_metrics[m_name] = v
                        
    return {'epochs': epochs, 'losses': losses, 'best_metrics': best_metrics}

# Quét log.txt của v1
v1_results = {}
log_files = glob.glob('/kaggle/working/STAIR-Enhanced/logs/**/log.txt', recursive=True)
for fpath in log_files:
    norm_p = fpath.replace('\\', '/').lower()
    if 'stair-v1' in norm_p or 'v1' in norm_p:
        parsed = parse_v1_log(fpath)
        if 'baby' in norm_p:
            v1_results['baby_v1'] = parsed
        elif 'sports' in norm_p:
            v1_results['sports_v1'] = parsed

# ---------------------------------------------------------
# 1. VẼ BIỂU ĐỒ QUÁ TRÌNH HỌC (TRAINING LOSS CURVES)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('STAIR-v1 Training Loss Curves', fontsize=16, fontweight='bold')

ds_keys = [('baby_v1', 'Baby Dataset', axes[0]), ('sports_v1', 'Sports Dataset', axes[1])]
for r_key, title, ax in ds_keys:
    data = v1_results.get(r_key, {})
    if data.get('epochs') and data.get('losses'):
        ax.plot(data['epochs'], data['losses'], color='#E74C3C', linewidth=1.8, label='STAIR-v1 Loss')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('BPR + CL Loss', fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Loss log currently training or empty', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v1_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# 2. BẢNG SO SÁNH TỔNG HỢP VÀ BIỂU ĐỒ CỘT METRICS
# ---------------------------------------------------------
final_metrics = {}
for k, v in BASELINE_METRICS.items():
    final_metrics[k] = v

for r_key in ['baby_v1', 'sports_v1']:
    final_metrics[r_key] = v1_results.get(r_key, {}).get('best_metrics', {})

print('\n' + '='*80)
print('BANG SO SANH CHIH SO: STAIR BASELINE VS STAIR-v1 (STAIR-GCL)')
print('='*80)
print(f"{'Metric':<12} | {'Baby Baseline':<15} | {'Baby STAIR-v1':<15} | {'Sports Baseline':<15} | {'Sports STAIR-v1':<15}")
print('-'*80)
for m in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
    b_b = final_metrics.get('baby_baseline', {}).get(m, 0.0)
    b_v = final_metrics.get('baby_v1', {}).get(m, 0.0)
    s_b = final_metrics.get('sports_baseline', {}).get(m, 0.0)
    s_v = final_metrics.get('sports_v1', {}).get(m, 0.0)
    print(f"{m:<12} | {b_b:<15.4f} | {b_v:<15.4f} | {s_b:<15.4f} | {s_v:<15.4f}")
print('='*80)

# Biểu đồ cột so sánh
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('STAIR Baseline vs STAIR-v1 (STAIR-GCL) Performance Comparison', fontsize=15, fontweight='bold')

datasets = ['baby', 'sports']
metric_keys = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    base_key = f'{ds}_baseline'
    v1_key   = f'{ds}_v1'
    
    base_vals = [final_metrics.get(base_key, {}).get(m, 0.0) for m in metric_keys]
    v1_vals   = [final_metrics.get(v1_key, {}).get(m, 0.0) for m in metric_keys]
    
    x = np.arange(len(metric_keys))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, base_vals, width, label='STAIR Baseline', color='#3498DB', alpha=0.85)
    bars2 = ax.bar(x + width/2, v1_vals, width, label='STAIR-v1 (GCL)', color='#2ECC71', alpha=0.85)
    
    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.4f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    
    ax.set_title(f'Dataset: {ds.upper()}', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_keys, fontsize=10)
    ax.set_ylabel('Metric Value')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved loss curve to: /kaggle/working/stair_v1_learning_curves.png')
print('Saved metrics chart to: /kaggle/working/stair_v1_comparison.png')